# Day 2 - Retrieval Optimization (AsthmaDaily RAG)
### GINA 2026 + WHO 2026 - Multi-Source Clinical RAG

Day 1 built a source-aware index that returns *something*. Today is about proving it returns the *right* thing - with real, measured numbers - and about picking which **embedding model** the project should actually use.

**By the end of this notebook you will be able to:**
1. Explain how `top_k` trades off precision against coverage
2. Run a controlled ablation experiment comparing chunk-size/overlap configurations
3. Compute Retrieval Precision@k, Hit Rate@k and MRR on a real, verified test set
4. Compare multiple embedding models on the exact same test set and read the report

> This notebook rebuilds the Day 1 index at the top so it's self-contained.


## 0. Setup - Rebuild the Day 1 Index

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config
from ingest import load_pdfs, preprocess, chunk_documents, build_index
from query import load_index, retrieve, source_aware_retrieve

pages = load_pdfs()
pages = preprocess(pages)
chunks = chunk_documents(pages)
vectordb = build_index(chunks)
print(f"\nIndex ready: {len(chunks)} chunks from {len(pages)} pages.")


## 1. What `top_k` Actually Controls

| `k` | Effect | Risk |
|---|---|---|
| Too low (1-2) | Very focused | Misses relevant evidence sitting in another section |
| Balanced (3-5) | Good coverage, manageable context | Usually the right starting point |
| Too high (10+) | Broad coverage | Dilutes context, invites irrelevant/contradictory chunks |

Let's see this directly on a real asthma question, run at three different `k` values.


In [ ]:
question = "What is the stepwise pharmacological approach to asthma treatment?"

for k in [1, 3, 8]:
    results = retrieve(vectordb, question, k=k, source="GINA")
    print(f"--- k={k} ---")
    for doc, score in results:
        print(f"  score={score:.3f}  page {doc.metadata.get('page_number')}: "
              f"{doc.page_content[:70].strip()}...")
    print()


### Checkpoint 1

Look at the `k=8` output. Are all 8 results still genuinely about the stepwise treatment approach, or do the later ones start drifting into other sections of the guideline? That drift is exactly why `top_k` needs to be tuned deliberately rather than set high "to be safe".


## 2. Ablation Experiment - Chunk Size & Overlap

Same source, same queries, only the chunking configuration changes.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ingest import get_embedding_function
from langchain_chroma import Chroma

test_queries = [
    "What inhaled corticosteroid doses are recommended for children with asthma?",
    "What are the risk factors for future asthma exacerbations?",
    "What is a written asthma action plan?",
]

configurations = [
    {"name": "Small (200/0)",     "chunk_size": 200, "chunk_overlap": 0},
    {"name": "Balanced (400/50)", "chunk_size": 400, "chunk_overlap": 50},
    {"name": "Large (600/100)",   "chunk_size": 600, "chunk_overlap": 100},
]

corpus_texts = [c.page_content for c in chunks]
embed_fn = get_embedding_function(corpus_texts=corpus_texts)
experiment_results = []

for cfg in configurations:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"] * config.CHARS_PER_TOKEN,
        chunk_overlap=cfg["chunk_overlap"] * config.CHARS_PER_TOKEN,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    test_chunks = splitter.split_documents(pages)
    test_db = Chroma.from_documents(
        documents=test_chunks, embedding=embed_fn,
        collection_name=f"experiment_{cfg['chunk_size']}",
    )

    avg_score = 0
    for q in test_queries:
        results = test_db.similarity_search_with_relevance_scores(q, k=3)
        avg_score += sum(s for _, s in results) / len(results)
    avg_score /= len(test_queries)

    experiment_results.append({"config": cfg["name"], "n_chunks": len(test_chunks), "avg_top3_score": avg_score})
    print(f"{cfg['name']:<20} chunks={len(test_chunks):>4}   avg top-3 relevance={avg_score:.3f}")


### Checkpoint 2

- Which configuration scored highest on average?
- Did the configuration with the *most* chunks also score the *best*? (It often doesn't.)

This project keeps **Balanced (400/50)** in `config.py` - large enough to keep a recommendation and its condition (e.g. an age group) in the same chunk, with a 50-token overlap so boundary content isn't lost, without diluting context the way the 600/100 config risks doing.


## 3. The Test Set

`eval/Day2_Evaluation_Test_Set.csv` has 12 real questions with verified expected sources: 10 answerable questions split across GINA and WHO, one deliberate out-of-scope control question, and one multi-source question that genuinely needs both documents.


In [ ]:
import csv

test_set = []
with open(config.EVAL_DIR / "Day2_Evaluation_Test_Set.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        test_set.append(row)

print(f"Loaded {len(test_set)} test questions.\n")
for row in test_set[:3]:
    print("Q:", row["Question"])
    print("   Expected:", row["Expected Source (Document / Section / Page)"])
    print()


## 4. Compute Precision@k, Hit Rate@k and MRR

$$\text{Precision@k} = \frac{\text{relevant chunks in top-k}}{k}$$

A retrieved chunk is scored as relevant if its `(document_name, page_number)` matches the expected source - a page-level, source-aware version of Precision@k (important here: page 26 in GINA and page 26 in WHO are different documents). `evaluate.py` implements this and adds **Hit Rate@k** (did at least one correct chunk show up at all) and **MRR** (how high did it rank).


In [ ]:
from evaluate import evaluate_model

metrics = evaluate_model(vectordb, k=3, verbose=True)

print("\n" + "-"*70)
print(f"Avg Precision@3: {metrics['avg_precision_at_k']:.3f}")
print(f"Hit Rate@3:      {metrics['hit_rate_at_k']:.3f}")
print(f"MRR:             {metrics['mrr']:.3f}")
print(f"Out-of-scope control question top score: {metrics['out_of_scope_control']['top_score']:.3f} "
      "(should be low - the index correctly has nothing relevant to return)")


### Checkpoint 3

- [ ] You have a real Precision@3, Hit Rate@3 and MRR - not a guess - for the current index
- [ ] The out-of-scope control question scored a low top similarity score
- [ ] `config.py`'s `CHUNK_SIZE` / `CHUNK_OVERLAP` reflect the configuration you're keeping


## 5. Embedding Model Comparison

This is the main addition for this project: instead of picking one embedding model on instinct, `config.EMBEDDING_MODELS` lists several candidates, and `run_pipeline.py` builds a **separate index per model** and evaluates all of them on the exact same test set:

| id | type | notes |
|---|---|---|
| `minilm-l6-v2` | HuggingFace | light general-purpose sentence embedding, common RAG default |
| `bge-small-en` | HuggingFace | retrieval-tuned model, usually stronger on question->passage retrieval |
| `gte-small`    | HuggingFace | third comparison point, competitive with bge-small |
| `tfidf-svd`    | offline (TF-IDF + LSA) | no downloads needed - validates the pipeline end-to-end, lower-bound baseline |

Run the full comparison from the terminal (`python run_pipeline.py`, from the project root) or inline here:


In [ ]:
# This mirrors run_pipeline.py's Step 4 - kept short here since the full run (with saved
# per-model reports) is in outputs/06_embedding_comparison_report.{json,md}
from evaluate import evaluate_model as _eval

comparison_rows = []
for model_config in config.EMBEDDING_MODELS:
    print(f"--- {model_config['id']} ---")
    try:
        idx = build_index(chunks, model_config=model_config)
        m = _eval(idx, k=3, verbose=False)
        comparison_rows.append((model_config["id"], m["avg_precision_at_k"], m["hit_rate_at_k"], m["mrr"]))
        print(f"  P@3={m['avg_precision_at_k']:.3f}  HitRate@3={m['hit_rate_at_k']:.3f}  MRR={m['mrr']:.3f}")
    except Exception as e:
        print(f"  skipped: {type(e).__name__}: {e}")


### Checkpoint 4 - Day 2 Self-Check

- [ ] You ran the same query at 3 different `k` values and can explain the trade-off out loud
- [ ] You ran the chunk-size/overlap ablation and can justify the configuration kept in `config.py`
- [ ] You compared at least 2 embedding models on the real test set (see `outputs/06_embedding_comparison_report.md` for the full, saved comparison and `README.md` for the write-up)
- [ ] You can name which embedding model this project uses by default (`config.DEFAULT_EMBEDDING_MODEL_ID`) and why

## What's Next

Day 3 (not built here) would constrain the LLM so tightly that every generated answer can only say what these retrieved, source-aware chunks actually support - with `document_name` + `page_number` citations, and an explicit GINA-vs-WHO split whenever the two guidelines differ for a child/adolescent question.
